# Lab: Generate Evaluation Data

This notebook generates realistic synthetic evaluation data for the dashboard lab.  
It produces `evaluation_data.csv` with columns:

| Column | Description |
|---|---|
| `evaluation_id` | Unique row identifier |
| `category` | Evaluation category (reasoning, knowledge, code, etc.) |
| `score` | Score 0–100 |
| `model_version` | Model name |
| `date` | Evaluation date (YYYY-MM-DD) |

In [1]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

## 1. Define model profiles

Each model has a different mean score per category, reflecting realistic performance differences.

In [2]:
# Categories and per-model mean scores (0–100)
CATEGORIES = [
    "reasoning",
    "knowledge",
    "code",
    "instruction_following",
    "tool_calling",
]

# model_name -> {category -> (mean, std)}
MODEL_PROFILES = {
    "model-v1": {
        "reasoning":             (62, 12),
        "knowledge":             (70, 10),
        "code":                  (55, 15),
        "instruction_following": (74, 9),
        "tool_calling":          (48, 16),
    },
    "model-v2": {
        "reasoning":             (71, 11),
        "knowledge":             (75, 9),
        "code":                  (68, 13),
        "instruction_following": (79, 8),
        "tool_calling":          (63, 14),
    },
    "model-v3": {
        "reasoning":             (80, 10),
        "knowledge":             (82, 8),
        "code":                  (77, 11),
        "instruction_following": (85, 7),
        "tool_calling":          (75, 13),
    },
}

N_PER_COMBO = 80   # evaluations per model × category combination
START_DATE  = date(2025, 1, 1)
END_DATE    = date(2025, 6, 1)

print(f"Models : {list(MODEL_PROFILES)}")
print(f"Categories : {CATEGORIES}")
print(f"Total rows : {len(MODEL_PROFILES) * len(CATEGORIES) * N_PER_COMBO:,}")

Models : ['model-v1', 'model-v2', 'model-v3']
Categories : ['reasoning', 'knowledge', 'code', 'instruction_following', 'tool_calling']
Total rows : 1,200


## 2. Generate rows

In [3]:
rows = []
eval_id = 1

date_range = (END_DATE - START_DATE).days

for model, profile in MODEL_PROFILES.items():
    for category in CATEGORIES:
        mean, std = profile[category]
        scores = rng.normal(mean, std, N_PER_COMBO)
        scores = np.clip(scores, 0, 100).round(1)

        for score in scores:
            random_day = rng.integers(0, date_range)
            eval_date  = START_DATE + timedelta(days=int(random_day))
            rows.append({
                "evaluation_id":  f"eval_{eval_id:04d}",
                "category":       category,
                "score":          score,
                "model_version":  model,
                "date":           eval_date.isoformat(),
            })
            eval_id += 1

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(df.shape)
df.head(10)

(1200, 5)


,evaluation_id,category,score,model_version,date
0,eval_1179,tool_calling,66.4,model-v3,2025-01-06
1,eval_0866,reasoning,95.2,model-v3,2025-01-29
2,eval_0102,knowledge,60.1,model-v1,2025-04-19
3,eval_0440,reasoning,99.0,model-v2,2025-04-09
4,eval_0059,reasoning,51.6,model-v1,2025-04-17
5,eval_1121,tool_calling,90.5,model-v3,2025-05-24
6,eval_0324,tool_calling,89.6,model-v1,2025-05-09
7,eval_0975,code,89.3,model-v3,2025-04-16
8,eval_0412,reasoning,64.1,model-v2,2025-02-12
9,eval_0856,reasoning,70.0,model-v3,2025-03-16


## 3. Sanity checks

In [4]:
print("=== Data types ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Score range ===")
print(df["score"].describe())

print("\n=== Rows per model × category ===")
print(df.groupby(["model_version", "category"])["score"].count().unstack())

=== Data types ===
evaluation_id        str
category             str
score            float64
model_version        str
date                 str
dtype: object

=== Missing values ===
evaluation_id    0
category         0
score            0
model_version    0
date             0
dtype: int64

=== Score range ===
count    1200.000000
mean       70.495083
std        14.799763
min         0.600000
25%        61.600000
50%        72.450000
75%        80.925000
max       100.000000
Name: score, dtype: float64

=== Rows per model × category ===
category       code  instruction_following  knowledge  reasoning  tool_calling
model_version                                                                 
model-v1         80                     80         80         80            80
model-v2         80                     80         80         80            80
model-v3         80                     80         80         80            80


## 4. Save to CSV

In [5]:
output_path = "evaluation_data.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows → {output_path}")

# Quick re-read check
df_check = pd.read_csv(output_path)
assert df_check.shape == df.shape, "Shape mismatch after re-read!"
print("Re-read check: OK")

Saved 1,200 rows → evaluation_data.csv
Re-read check: OK
